# 01 — Inventario de fuentes y diccionario de datos

**Etapa CRISP-DM:** Comprensión de datos · **Semana 5 del cronograma** · **OE2**

## Objetivo
Caracterizar las fuentes de datos entregadas por la Universidad: número de
registros, variables, granularidad (unidad de información), claves de unión,
periodos cubiertos y pertinencia frente a los objetivos del proyecto. El
producto es un **diccionario de datos preliminar** que servirá de base para las
etapas de calidad (semana 6) e integración (semanas 7-8).

## Contexto
El proyecto busca analizar la participación estudiantil en internacionalización.
Las fuentes son archivos `parquet` anonimizados ubicados en `data/`. No existe
una tabla explícita de "experiencias de internacionalización": esta señal deberá
**derivarse** de los registros académicos (se aborda en el notebook 03).

In [1]:
import sys, os
from pathlib import Path
# Permitir importar el paquete src desde notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.titlesize'] = 12

from src import config, utils, internationalization as intl
print('Entorno listo. Raiz del proyecto:', ROOT)

Entorno listo. Raiz del proyecto: C:\Users\JUAN DAVID\Documents\UNIANDES\8VO SEMESTRE\PROY GRADO 2026-2\TRAVAIL\PROY-GRADO


## 1. Carga del inventario de fuentes
Recorremos las fuentes registradas en `config.SOURCES` y describimos su tamaño.

In [2]:
import os
rows = []
frames = {}
for name, fname in config.SOURCES.items():
    path = config.DATA_DIR / fname
    df = utils.load_source(name)
    frames[name] = df
    rows.append({
        'fuente': name,
        'archivo': fname,
        'filas': len(df),
        'columnas': df.shape[1],
        'tamano_MB': round(os.path.getsize(path) / 1e6, 1),
    })
inventario = pd.DataFrame(rows).sort_values('filas', ascending=False).reset_index(drop=True)
inventario

,fuente,archivo,filas,columnas,tamano_MB
0,momento_academico,momento_academico_materia_estudiante_anonimiza...,6983336,11,33.0
1,historial_materias,historial_materias_estudiante_anonimizacio.par...,6958732,14,75.6
2,programas,programas_estudiante_anonimizacion.parquet,1770486,18,15.1
3,historial_rendimiento,historial_rendimiento_academico_estudiante_ano...,1230195,28,45.8
4,percentiles_historicos,percentiles_historicos_academicos_estudiante_a...,994875,18,23.2
5,icfes_saber11,icfes_saber11_estudiante_anonimizacion.parquet,262148,10,5.4
6,informacion_actual,informacion_actual_estudiante_anonimizacion.pa...,233987,73,12.1
7,percentiles_academicos,percentiles_academicos_estudiante_anonimizacio...,195276,10,3.9
8,historial_opciones,historial_opciones_estudiante_anonimizacion.pa...,173387,14,1.1
9,grados,grados_estudiante_anonimizacion.parquet,139429,10,2.5


### Resultado e interpretación
El universo de datos combina fuentes de muy distinta escala. Las fuentes
**transaccionales** (una fila por evento académico) son las más grandes:
`historial_materias` (~6.9 M filas, un registro por curso-estudiante-periodo),
`programas` (~1.8 M) e `historial_rendimiento` (~1.2 M, un registro por
estudiante-periodo). Las fuentes **de estudiante** son más pequeñas:
`informacion_actual` (~234 k filas) es un snapshot con **una fila por
estudiante**, lo que la convierte en la candidata natural a *columna vertebral*
de la unidad de análisis.

## 2. Identificación de la clave de estudiante y granularidad
Verificamos qué columna identifica al estudiante y si cada fuente es a nivel estudiante o transaccional.

In [3]:
def student_col(df):
    for c in (config.STUDENT_KEY, config.STUDENT_KEY_ANOM):
        if c in df.columns:
            return c
    return None

gran = []
for name, df in frames.items():
    key = student_col(df)
    n = len(df)
    n_est = df[key].nunique() if key else np.nan
    filas_por_est = round(n / n_est, 2) if key and n_est else np.nan
    gran.append({
        'fuente': name,
        'clave_estudiante': key,
        'n_estudiantes': n_est,
        'filas': n,
        'filas_por_estudiante': filas_por_est,
        'unidad': 'estudiante' if filas_por_est == 1 else 'transaccional/otra',
    })
granularidad = pd.DataFrame(gran).sort_values('filas_por_estudiante', ascending=False)
granularidad

,fuente,clave_estudiante,n_estudiantes,filas,filas_por_estudiante,unidad
8,momento_academico,CODIGO_ESTUDIANTE,188346,6983336,37.08,transaccional/otra
3,historial_materias,CODIGO_ESTUDIANTE_ANOM,188436,6958732,36.93,transaccional/otra
4,historial_opciones,CODIGO_ESTUDIANTE,21601,173387,8.03,transaccional/otra
10,percentiles_historicos,CODIGO_ESTUDIANTE,126750,994875,7.85,transaccional/otra
5,historial_rendimiento,CODIGO_ESTUDIANTE,177527,1230195,6.93,transaccional/otra
11,programas,CODIGO_ESTUDIANTE_ANOM,287446,1770486,6.16,transaccional/otra
9,percentiles_academicos,CODIGO_ESTUDIANTE,148854,195276,1.31,transaccional/otra
2,grados,CODIGO_ESTUDIANTE,112541,139429,1.24,transaccional/otra
0,capp_creditos,CODIGO_ESTUDIANTE,25449,30771,1.21,transaccional/otra
6,icfes_saber11,CODIGO_ESTUDIANTE,244066,262148,1.07,transaccional/otra


### Resultado e interpretación
Todas las fuentes identifican al estudiante mediante un código anonimizado:
la mayoría usa `CODIGO_ESTUDIANTE`, mientras que `historial_materias` y
`programas` usan `CODIGO_ESTUDIANTE_ANOM`. Esta diferencia de nombre es un primer
problema de integración que se normaliza en el pipeline (semana 7).

`informacion_actual` tiene exactamente **1 fila por estudiante** (granularidad de
estudiante); el resto son fuentes transaccionales con varias filas por
estudiante. **Decisión:** se adopta el estudiante como unidad de análisis y
`informacion_actual` como columna vertebral; las fuentes transaccionales se
agregarán a nivel estudiante.

## 3. Periodos cubiertos
Los periodos tienen formato `AAAASS`. Revisamos el rango temporal de las fuentes con periodo.

In [4]:
periodos = []
for name, df in frames.items():
    pcol = next((c for c in ['PERIODO', 'PERIODO_SNAPSHOT'] if c in df.columns), None)
    if pcol is None:
        continue
    anios = df[pcol].map(utils.periodo_to_year).dropna()
    if len(anios):
        periodos.append({'fuente': name, 'anio_min': int(anios.min()),
                         'anio_max': int(anios.max())})
periodos_df = pd.DataFrame(periodos).sort_values('anio_min')
periodos_df

,fuente,anio_min,anio_max
9,programas,0,2027
1,historial_materias,520,9999
6,momento_academico,520,9999
0,grados,1946,2026
5,informacion_actual,1948,2027
4,icfes_saber11,1948,9999
7,percentiles_academicos,1961,2026
3,historial_rendimiento,1961,2026
2,historial_opciones,2000,2027
8,percentiles_historicos,2001,2026


### Resultado e interpretación
Las fuentes transaccionales cubren un rango histórico amplio (desde finales de
los años 1990/2000 hasta 2026), suficiente para análisis de tendencias
temporales de largo plazo (PA1). El snapshot `informacion_actual` registra el
último periodo observado de cada estudiante.

## 4. Diccionario de datos preliminar
Generamos una descripción columna a columna de todas las fuentes y la exportamos a `docs/`.

In [5]:
dicc_parts = []
for name, df in frames.items():
    prof = utils.profile_dataframe(df, name=name)
    dicc_parts.append(prof)
diccionario = pd.concat(dicc_parts, ignore_index=True)
print('Total de columnas documentadas:', len(diccionario))
# Exportar diccionario completo
out_csv = config.TABLES_DIR / 'diccionario_datos.csv'
diccionario.to_csv(out_csv, index=False, encoding='utf-8')
print('Diccionario exportado a', out_csv)
diccionario.head(20)

Total de columnas documentadas: 221
Diccionario exportado a C:\Users\JUAN DAVID\Documents\UNIANDES\8VO SEMESTRE\PROY GRADO 2026-2\TRAVAIL\PROY-GRADO\reports\tables\diccionario_datos.csv


,fuente,columna,dtype,no_nulos,faltantes,pct_faltantes,n_unicos,cardinalidad_pct,ejemplo
0,capp_creditos,CODIGO_ESTUDIANTE,object,30771,0,0.00,25449.0,82.70,b232e9cab9c6
1,capp_creditos,PROGRAMA,object,30771,0,0.00,50.0,0.16,DISO
2,capp_creditos,CREDITOS_REQUERIDOS,float64,30707,64,0.21,30.0,0.10,134.0
3,capp_creditos,CREDITOS_COMPLETADOS,float64,30707,64,0.21,326.0,1.06,10.0
4,capp_creditos,CREDITOS_FALTANTES,float64,30707,64,0.21,324.0,1.05,124.0
5,capp_creditos,REQUISITOS_CUMPLIDOS,bool,30771,0,0.00,2.0,0.01,False
6,capp_creditos,FECHA_CALCULO_CAPP,datetime64[ns],30771,0,0.00,314.0,1.02,2026-07-17 06:57:08.910812
7,examenes_clasificacion,CODIGO_ESTUDIANTE,object,79599,0,0.00,78511.0,98.63,08dee4b3c320
8,examenes_clasificacion,NIVEL_ACADEMICO,object,79599,0,0.00,3.0,0.00,Posgrado
9,examenes_clasificacion,NIVEL_INGLES,float64,36034,43565,54.73,8.0,0.01,2.0


### Variables candidatas por bloque temático
Agrupamos las variables que, por la propuesta, son pertinentes para el análisis.

In [6]:
bloques = {
    'Identificación / demografía': ['CODIGO_ESTUDIANTE', 'EDAD', 'GENERO',
        'ESTRATO_SOCIAL', 'DEPARTAMENTO_COLEGIO', 'CIUDAD_COLEGIO'],
    'Programa / unidad académica': ['CODIGO_FACULTAD_PROGRAMA_1', 'PROGRAMA_1',
        'NIVEL_PROGRAMA_1', 'TIPO_ADMISION'],
    'Desempeño académico': ['PGA', 'PROMEDIO_SEMESTRAL', 'CREDITOS_APROBADOS',
        'TOTAL_SEMESTRES_MATRICULADOS', 'PERCENTIL_PGA_PROGRAMA', 'SEMESTRE_AVANCE'],
    'Pruebas de ingreso': ['PUNTAJE_TOTAL', 'MATEMATICAS', 'LECTURA_CRITICA',
        'INGLES', 'NIVEL_INGLES'],
    'Señal de internacionalización (derivada)': ['NOMBRE_CURSO', 'CODIGO_CURSO',
        'ESTATUS', 'CODIGO_CAMPUS'],
}
for b, cols in bloques.items():
    presentes = sorted(set(cols) & set(diccionario['columna']))
    print(f"\n{b}:")
    print('  ', ', '.join(presentes))


Identificación / demografía:
   CIUDAD_COLEGIO, CODIGO_ESTUDIANTE, DEPARTAMENTO_COLEGIO, EDAD, ESTRATO_SOCIAL, GENERO

Programa / unidad académica:
   CODIGO_FACULTAD_PROGRAMA_1, NIVEL_PROGRAMA_1, PROGRAMA_1, TIPO_ADMISION

Desempeño académico:
   CREDITOS_APROBADOS, PERCENTIL_PGA_PROGRAMA, PGA, PROMEDIO_SEMESTRAL, SEMESTRE_AVANCE, TOTAL_SEMESTRES_MATRICULADOS

Pruebas de ingreso:
   INGLES, LECTURA_CRITICA, MATEMATICAS, NIVEL_INGLES, PUNTAJE_TOTAL

Señal de internacionalización (derivada):
   CODIGO_CAMPUS, CODIGO_CURSO, ESTATUS, NOMBRE_CURSO


## 5. Conclusiones y decisiones (semana 5)

- **Unidad de análisis:** el estudiante. `informacion_actual` (1 fila/estudiante)
  es la columna vertebral.
- **Claves de unión:** `CODIGO_ESTUDIANTE`, con normalización de
  `CODIGO_ESTUDIANTE_ANOM` → `CODIGO_ESTUDIANTE` en `historial_materias` y
  `programas`.
- **Fuentes núcleo** (confirmadas pertinentes): `informacion_actual`,
  `historial_materias` (fuente de la señal de internacionalización),
  `historial_rendimiento`, `percentiles_historicos`, `icfes_saber11`,
  `examenes_clasificacion`, `capp_creditos`.
- **Problema detectado:** las columnas de texto presentan *mojibake* de
  codificación (p. ej. `Mag¡ster`, `Educaci¾n`). Se analiza y corrige en el
  notebook 02.
- El diccionario de datos preliminar queda exportado en
  `reports/tables/diccionario_datos.csv`.

**Siguiente paso:** perfilamiento y evaluación de calidad de datos (semana 6).